# NeuroRouter API Testing

Testing the NeuroRouter AWS backend with a real API key.
This verifies that:
1. API key authentication works
2. LLM inference works (Groq backend via AWS Lambda)
3. Token usage is recorded in DynamoDB
4. Dashboard reflects the usage

In [19]:
import openai
import json

# NeuroRouter AWS backend
API_BASE = "https://u87jos3lg5.execute-api.ap-south-1.amazonaws.com/dev"

# Create an API key from the dashboard first (http://localhost:3000/dashboard/api-keys)
# Then paste it here:
API_KEY = "neurorouter_U0g4X6Yy1oFx3"  #neurorouter_Kks8EyqGq813P e.g. "neurorouter_xxxxxxxxx"

client = openai.OpenAI(
    base_url=f"{API_BASE}/v1",
    api_key=API_KEY,
)

print(f"Connected to: {API_BASE}")
print(f"API Key: {API_KEY[:20]}...")

Connected to: https://u87jos3lg5.execute-api.ap-south-1.amazonaws.com/dev
API Key: neurorouter_U0g4X6Yy...


## Test 1: List Available Models

In [2]:
models = client.models.list()
print(f"Available models: {len(models.data)}")
for m in models.data:
    print(f"  - {m.id} (owned by: {m.owned_by})")

Available models: 5
  - llama-3.3-70b-versatile (owned by: groq)
  - llama-3.1-8b-instant (owned by: groq)
  - llama-3.1-70b-versatile (owned by: groq)
  - mixtral-8x7b-32768 (owned by: groq)
  - gemma2-9b-it (owned by: groq)


## Test 2: Simple Chat Completion

In [3]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "What is NeuroRouter? Answer in 2 sentences."}
    ],
    max_tokens=100,
)

print("AI Response:")
print(response.choices[0].message.content)
print(f"\nTokens used: prompt={response.usage.prompt_tokens}, completion={response.usage.completion_tokens}, total={response.usage.total_tokens}")

AI Response:
NeuroRouter is an open-source software framework that enables the creation of brain-computer interfaces (BCIs) and other neurotechnology applications, allowing users to control devices with their brain signals. It provides a flexible and modular architecture for processing and analyzing neural data, making it a useful tool for researchers, developers, and practitioners in the field of neurotechnology and neuroscience.

Tokens used: prompt=46, completion=74, total=120


## Test 3: Code Generation

In [4]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are a Python expert. Write clean, concise code."},
        {"role": "user", "content": "Write a Python function that calculates the monthly billing cost given input_tokens and output_tokens. Free tier: 1M each. Overage: $2/1M input, $8/1M output. Fixed fee: Rs 1599."}
    ],
    max_tokens=500,
)

print("Code Generation:")
print(response.choices[0].message.content)
print(f"\nTokens: prompt={response.usage.prompt_tokens}, completion={response.usage.completion_tokens}")

Code Generation:
```python
def calculate_monthly_billing_cost(input_tokens, output_tokens):
    """
    Calculate the monthly billing cost given input_tokens and output_tokens.

    Parameters:
    input_tokens (int): The number of input tokens.
    output_tokens (int): The number of output tokens.

    Returns:
    float: The monthly billing cost.
    """
    # Define the free tier limits and overage costs
    free_tier_limit = 1000000  # 1M tokens
    input_overage_cost = 2  # $2 per 1M input tokens
    output_overage_cost = 8  # $8 per 1M output tokens
    fixed_fee = 1599  # Rs 1599 fixed fee

    # Calculate the overage for input tokens
    input_overage = max(0, (input_tokens - free_tier_limit) / 1000000)

    # Calculate the overage for output tokens
    output_overage = max(0, (output_tokens - free_tier_limit) / 1000000)

    # Calculate the total overage cost
    total_overage_cost = (input_overage * input_overage_cost) + (output_overage * output_overage_cost)

    # Calculate

## Test 4: Multi-turn Conversation

In [5]:
messages = [
    {"role": "system", "content": "You are a helpful assistant for a cloud billing platform."},
    {"role": "user", "content": "I used 3.5M input tokens and 1.2M output tokens this month. What's my bill?"}
]

# Turn 1
r1 = client.chat.completions.create(model="llama-3.3-70b-versatile", messages=messages, max_tokens=300)
assistant_reply = r1.choices[0].message.content
print(f"Turn 1: {assistant_reply}")
print(f"Tokens: {r1.usage.total_tokens}")

# Turn 2 - follow up
messages.append({"role": "assistant", "content": assistant_reply})
messages.append({"role": "user", "content": "What if I upgrade and my output rate drops to $6/1M?"})

r2 = client.chat.completions.create(model="llama-3.3-70b-versatile", messages=messages, max_tokens=300)
print(f"\nTurn 2: {r2.choices[0].message.content}")
print(f"Tokens: {r2.usage.total_tokens}")

Turn 1: To calculate your bill, I'll need to know the pricing details for input and output tokens. Our standard pricing is $0.000004 per input token and $0.000008 per output token. 

For input tokens: 3,500,000 * $0.000004 = $14
For output tokens: 1,200,000 * $0.000008 = $9.60

Your total bill would be $14 + $9.60 = $23.60. Does that sound correct to you?
Tokens: 181

Turn 2: If your output rate drops to $6 per 1 million tokens, that's a significant discount. 

For input tokens, the pricing remains the same: 3,500,000 * $0.000004 = $14

For output tokens, with the new rate: 1,200,000 tokens / 1,000,000 = 1.2, and 1.2 * $6 = $7.20

Your new total bill would be $14 + $7.20 = $21.20. You'd be saving $2.40 compared to your previous bill. Would you like me to explain the upgrade process or answer any other questions?
Tokens: 345


## Test 5: Multiple Rapid Requests (Usage Accumulation)

In [6]:
total_prompt = 0
total_completion = 0

prompts = [
    "What is DynamoDB?",
    "Explain AWS Lambda in one sentence.",
    "What is API Gateway?",
    "Explain Cognito briefly.",
    "What is S3?",
]

for i, prompt in enumerate(prompts):
    r = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=50,
    )
    total_prompt += r.usage.prompt_tokens
    total_completion += r.usage.completion_tokens
    print(f"  [{i+1}] {prompt[:40]:40s} → {r.usage.total_tokens} tokens")

print(f"\n=== TOTAL: prompt={total_prompt}, completion={total_completion}, total={total_prompt + total_completion} ===")
print("\nCheck your dashboard to see these tokens reflected in the usage chart.")

  [1] What is DynamoDB?                        → 90 tokens
  [2] Explain AWS Lambda in one sentence.      → 88 tokens
  [3] What is API Gateway?                     → 90 tokens
  [4] Explain Cognito briefly.                 → 91 tokens
  [5] What is S3?                              → 90 tokens

=== TOTAL: prompt=204, completion=245, total=449 ===

Check your dashboard to see these tokens reflected in the usage chart.


## Interactive Chat — Enter Your Own Prompts

Run the cell below to start an interactive chat session. Type your prompts and get AI responses in real-time. Type `quit` or `exit` to stop.

In [24]:
conversation = [
    {"role": "system", "content": "You are a helpful AI assistant powered by NeuroRouter."}
]
total_tokens_used = 0

print("=== NeuroRouter Interactive Chat ===")
print("Type your message and press Enter. Type 'quit' to exit.\n")

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ("quit", "exit", "q"):
        print(f"\n--- Session ended. Total tokens used: {total_tokens_used} ---")
        break
    
    conversation.append({"role": "user", "content": user_input})
    
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=conversation,
            max_tokens=500,
        )
        
        reply = response.choices[0].message.content
        tokens = response.usage.total_tokens
        total_tokens_used += tokens
        
        conversation.append({"role": "assistant", "content": reply})
        
        print(f"\nAI: {reply}")
        print(f"   [{tokens} tokens | session total: {total_tokens_used}]\n")
        
    except Exception as e:
        print(f"\nError: {e}\n")
        conversation.pop()  # remove failed user message

=== NeuroRouter Interactive Chat ===
Type your message and press Enter. Type 'quit' to exit.


AI: Consonants are letters in the alphabet that are not vowels. In the English language, consonants are all the letters that are not A, E, I, O, or U. Sometimes, Y is also considered a consonant.

Consonants are sounds that are made by obstructing airflow with the tongue, teeth, or lips. They can be classified into several categories, including:

1. Stops: Sounds like /p/, /t/, /k/, /b/, /d/, and /g/, which are made by blocking the airflow with the tongue or lips.
2. Fricatives: Sounds like /f/, /v/, /s/, /z/, /sh/, and /th/, which are made by directing airflow through a narrow channel.
3. Nasals: Sounds like /m/, /n/, and /ng/, which are made by directing airflow through the nose.
4. Liquids: Sounds like /l/ and /r/, which are made by allowing airflow to pass through the tongue and lips.
5. Semivowels: Sounds like /w/, /y/, and /h/, which are made by modifying the sound of a vowel.

Consonan

## Test 6: Verify Dashboard Shows Usage

After running the tests above, open these pages to verify:

1. **http://localhost:3000/dashboard** → Total Tokens and Total Requests should increase
2. **http://localhost:3000/dashboard/usage** → Usage chart should show tokens for this month
3. **http://localhost:3000/dashboard/billing** → Current cycle should show token counts and estimated cost

## Demo: Token Limit Warning Banners

Run the cells below to simulate usage levels and see the warning banners on the dashboard.
- **Cell A**: Sets usage to 85% (Yellow warning — "Approaching Free Tier Limit")
- **Cell B**: Sets usage to 120% (Red warning — "Free Tier Limit Exceeded")

After running each cell, refresh `http://localhost:3000/dashboard` to see the banner.

In [20]:
# DEMO CELL A: Set usage to 85% — triggers YELLOW warning banner
# "Approaching Free Tier Limit" with amber progress bars
import boto3, requests

dynamodb = boto3.client('dynamodb', region_name='ap-south-1')

# Login to get user ID
login = requests.post(f"{API_BASE}/auth/login", json={"email": "test3@gmail.com", "password": "Abc123456789"})
if login.status_code != 200 or "access_token" not in login.json():
    raise Exception(f"Login failed: {login.text}")
jwt_token = login.json()["access_token"]
me = requests.get(f"{API_BASE}/auth/me", headers={"Authorization": f"Bearer {jwt_token}"})
user_id = me.json()["userId"]
print(f"User: {me.json()['email']} (ID: {user_id})")

# Set input to 850k (85%) and output to 750k (75%)
dynamodb.put_item(
    TableName='neurorouter-usage-monthly-dev',
    Item={
        'userId': {'S': user_id},
        'sk': {'S': '2026-04#MODEL#llama-3.3-70b-versatile#KEY#demo-limit-test'},
        'year_month': {'S': '2026-04'},
        'model': {'S': 'llama-3.3-70b-versatile'},
        'api_key_id': {'S': 'demo-limit-test'},
        'input_tokens': {'N': '850000'},
        'output_tokens': {'N': '750000'},
        'total_tokens': {'N': '1600000'},
        'request_count': {'N': '500'},
    }
)

print("\n--- Usage set to 85% input / 75% output ---")
print("   Input:  850,000 / 1,000,000 (85%) -> YELLOW")
print("   Output: 750,000 / 1,000,000 (75%) -> normal")
print("\n>>> Refresh http://localhost:3000/dashboard to see the YELLOW warning banner")


User: test3@gmail.com (ID: 31f38d1a-5071-70ac-3751-c976949af270)

--- Usage set to 85% input / 75% output ---
   Input:  850,000 / 1,000,000 (85%) -> YELLOW
   Output: 750,000 / 1,000,000 (75%) -> normal

>>> Refresh http://localhost:3000/dashboard to see the YELLOW warning banner


In [ ]:
# DEMO CELL B: Set usage to 120% — triggers RED exceeded banner
# "Free Tier Limit Exceeded" with red progress bars and overage message
import boto3, requests

dynamodb = boto3.client('dynamodb', region_name='ap-south-1')

# Login to get user ID
login = requests.post(f"{API_BASE}/auth/login", json={"email": "test3@gmail.com", "password": "Abc123456789"})
if login.status_code != 200 or "access_token" not in login.json():
    raise Exception(f"Login failed: {login.text}")
jwt_token = login.json()["access_token"]
me = requests.get(f"{API_BASE}/auth/me", headers={"Authorization": f"Bearer {jwt_token}"})
user_id = me.json()["userId"]
print(f"User: {me.json()['email']} (ID: {user_id})")

dynamodb.put_item(
    TableName='neurorouter-usage-monthly-dev',
    Item={
        'userId': {'S': user_id},
        'sk': {'S': '2026-04#MODEL#llama-3.3-70b-versatile#KEY#demo-limit-test'},
        'year_month': {'S': '2026-04'},
        'model': {'S': 'llama-3.3-70b-versatile'},
        'api_key_id': {'S': 'demo-limit-test'},
        'input_tokens': {'N': '1200000'},
        'output_tokens': {'N': '1100000'},
        'total_tokens': {'N': '2300000'},
        'request_count': {'N': '800'},
    }
)

print("--- Usage set to 120% input / 110% output ---")
print("   Input:  1,200,000 / 1,000,000 (120%) -> RED")
print("   Output: 1,100,000 / 1,000,000 (110%) -> RED")
print("   Overage: +200k input ($0.40) + 100k output ($0.80)")
print("\n>>> Refresh http://localhost:3000/dashboard to see the RED exceeded banner")


User: yoyoharishankar@gmail.com (ID: 21935d6a-f041-70f6-0adf-1f37f36d39b1)
--- Usage set to 120% input / 110% output ---
   Input:  1,200,000 / 1,000,000 (120%) -> RED
   Output: 1,100,000 / 1,000,000 (110%) -> RED
   Overage: +200k input ($0.40) + 100k output ($0.80)

>>> Refresh http://localhost:3000/dashboard to see the RED exceeded banner


In [23]:
# DEMO CELL C: Reset usage back to normal (100k tokens)
# Clears warning banners from the dashboard
import boto3, requests

dynamodb = boto3.client('dynamodb', region_name='ap-south-1')

# Login to get user ID
login = requests.post(f"{API_BASE}/auth/login", json={"email": "test3@gmail.com", "password": "Abc123456789"})
if login.status_code != 200 or "access_token" not in login.json():
    raise Exception(f"Login failed: {login.text}")
jwt_token = login.json()["access_token"]
me = requests.get(f"{API_BASE}/auth/me", headers={"Authorization": f"Bearer {jwt_token}"})
user_id = me.json()["userId"]
print(f"User: {me.json()['email']} (ID: {user_id})")

# Delete all existing usage rows for this user
scan = dynamodb.query(
    TableName='neurorouter-usage-monthly-dev',
    KeyConditionExpression='userId = :uid',
    ExpressionAttributeValues={':uid': {'S': user_id}}
)
for item in scan.get('Items', []):
    dynamodb.delete_item(
        TableName='neurorouter-usage-monthly-dev',
        Key={'userId': item['userId'], 'sk': item['sk']}
    )
print(f"Deleted {len(scan.get('Items', []))} existing usage rows")

# Insert a single clean row with 100k tokens
dynamodb.put_item(
    TableName='neurorouter-usage-monthly-dev',
    Item={
        'userId': {'S': user_id},
        'sk': {'S': '2026-04#MODEL#llama-3.3-70b-versatile#KEY#demo-limit-test'},
        'year_month': {'S': '2026-04'},
        'model': {'S': 'llama-3.3-70b-versatile'},
        'api_key_id': {'S': 'demo-limit-test'},
        'input_tokens': {'N': '0'},
        'output_tokens': {'N': '0'},
        'total_tokens': {'N': '0'},
        'request_count': {'N': '50'},
    }
)

print("\n--- Usage reset to 100k tokens ---")
print("   Input:  50,000 / 1,000,000 (5%) -> normal")
print("   Output: 50,000 / 1,000,000 (5%) -> normal")
print("\n>>> Refresh http://localhost:3000/dashboard — no warning banner")


User: test3@gmail.com (ID: 31f38d1a-5071-70ac-3751-c976949af270)
Deleted 2 existing usage rows

--- Usage reset to 100k tokens ---
   Input:  50,000 / 1,000,000 (5%) -> normal
   Output: 50,000 / 1,000,000 (5%) -> normal

>>> Refresh http://localhost:3000/dashboard — no warning banner
